# benchmarking_atari_200m

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"
REFERENCE_CONFIG = Path("../atari_200m_deterministic/config.py")

In [ ]:
# Load Agent0's and Agent1's stored runs per game, and the DQN runs of
# atari_200m_deterministic as the reference.
AGENT0 = "Agent0"
AGENT1 = "Agent1"
DQN = "DQN"
component = EXPERIMENT.component("agent0_atari")
GAMES = component.sweep["ENV_HYPERS.GAME"]
FRAMESKIP = component.config.ENV_HYPERS.FRAMESKIP
TOTAL_FRAMES = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS * FRAMESKIP

spec = importlib.util.spec_from_file_location("reference_config", REFERENCE_CONFIG)
reference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reference)
STORES = {
    AGENT0: (EXPERIMENT, "agent0_atari"),
    AGENT1: (EXPERIMENT, "agent1_atari"),
    DQN: (reference.EXPERIMENT, "dqn_atari"),
}

results = {}
for agent, (experiment, name) in STORES.items():
    for game in GAMES:
        runs = load_results(experiment, name, where={"ENV_HYPERS.GAME": game})
        if runs is not None:
            results[agent, game] = runs

In [ ]:
# Compute each agent's mean return per game with a 95% bootstrap CI.
curves = {}
for (agent, game), runs in results.items():
    timesteps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[agent, game] = (timesteps * FRAMESKIP, mean_ci(returns))

In [ ]:
# Style for the all-games grid.
LINES = {
    DQN: {"color": "tab:blue", "linewidth": 1.75, "label": "DQN"},
    AGENT0: {"color": "tab:red", "linewidth": 1.75, "label": "Agent0"},
    AGENT1: {"color": "tab:green", "linewidth": 1.75, "label": "Agent1"},
}
BAND_ALPHA = 0.2
GRID_STYLE = {
    "figure.figsize": (20, 49),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "figure.labelsize": 20,
}
GRID_SHAPE = (13, 5)
PANELS = {  # (column, row) of each game's panel
    "air_raid": (0, 0),
    "alien": (1, 0),
    "amidar": (2, 0),
    "assault": (3, 0),
    "asterix": (4, 0),
    "asteroids": (0, 1),
    "atlantis": (1, 1),
    "bank_heist": (2, 1),
    "battle_zone": (3, 1),
    "beam_rider": (4, 1),
    "berzerk": (0, 2),
    "bowling": (1, 2),
    "boxing": (2, 2),
    "breakout": (3, 2),
    "carnival": (4, 2),
    "centipede": (0, 3),
    "chopper_command": (1, 3),
    "crazy_climber": (2, 3),
    "defender": (3, 3),
    "demon_attack": (4, 3),
    "double_dunk": (0, 4),
    "elevator_action": (1, 4),
    "enduro": (2, 4),
    "fishing_derby": (3, 4),
    "freeway": (4, 4),
    "frogger": (0, 5),
    "frostbite": (1, 5),
    "gopher": (2, 5),
    "gravitar": (3, 5),
    "hero": (4, 5),
    "ice_hockey": (0, 6),
    "jamesbond": (1, 6),
    "journey_escape": (2, 6),
    "kangaroo": (3, 6),
    "krull": (4, 6),
    "kung_fu_master": (0, 7),
    "montezuma_revenge": (1, 7),
    "ms_pacman": (2, 7),
    "name_this_game": (3, 7),
    "phoenix": (4, 7),
    "pitfall": (0, 8),
    "pong": (1, 8),
    "pooyan": (2, 8),
    "private_eye": (3, 8),
    "qbert": (4, 8),
    "riverraid": (0, 9),
    "road_runner": (1, 9),
    "robotank": (2, 9),
    "seaquest": (3, 9),
    "skiing": (4, 9),
    "solaris": (0, 10),
    "space_invaders": (1, 10),
    "star_gunner": (2, 10),
    "surround": (3, 10),
    "tennis": (4, 10),
    "tetris": (0, 11),
    "time_pilot": (1, 11),
    "tutankham": (2, 11),
    "up_n_down": (3, 11),
    "venture": (4, 11),
    "video_pinball": (0, 12),
    "wizard_of_wor": (1, 12),
    "yars_revenge": (2, 12),
    "zaxxon": (3, 12),
}
PANEL_ASPECT = 3 / 4
GRID_XTICKS = [0, TOTAL_FRAMES]
GRID_XTICK_LABELS = ["0", f"{TOTAL_FRAMES / 1_000_000:.0f}M"]
GRID_LAYOUT = {"rect": (0.03, 0.0175, 1, 1), "w_pad": 0.2, "h_pad": 0.2}
GRID_XLABEL = {"t": "Frames", "y": 0.0165, "va": "top"}
GRID_YLABEL = {"t": "Return"}
GRID_LEGEND = {"loc": "upper center", "bbox_to_anchor": (0.5, 0.0083), "ncol": 3}


def title(game):
    return game.replace("_", " ").title()


def draw(ax, agent, game):
    frames, (mean, low, high) = curves[agent, game]
    line = LINES[agent]
    ax.fill_between(frames, low, high, color=line["color"], alpha=BAND_ALPHA)
    ax.plot(frames, mean, **line)

In [ ]:
# Plot every game in one grid: Agent0 and Agent1 against the DQN reference.
with plt.rc_context(GRID_STYLE):
    fig, axes = plt.subplots(*GRID_SHAPE)
    panel_axes = {axes[row, column] for column, row in PANELS.values()}
    for ax in set(axes.flat) - panel_axes:
        ax.axis("off")
    legend = {}
    for game, (column, row) in PANELS.items():
        ax = axes[row, column]
        ax.set_box_aspect(PANEL_ASPECT)
        drawn = [agent for agent in LINES if (agent, game) in curves]
        if not drawn:
            ax.axis("off")
            continue
        for agent in drawn:
            draw(ax, agent, game)
        ax.set_title(title(game))
        ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
        handles, labels = ax.get_legend_handles_labels()
        legend.update(zip(labels, handles, strict=True))
    fig.tight_layout(**GRID_LAYOUT)
    fig.supxlabel(**GRID_XLABEL)
    fig.supylabel(**GRID_YLABEL)
    fig.legend(legend.values(), legend.keys(), **GRID_LEGEND)
figures = {"grid": fig}
plt.show()

In [ ]:
# Save every plotted figure as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")